# Practical-11: Propositional & First-Order Logic Inference

**Aim:** To represent a real-world problem (scholarship eligibility) using propositional and first-order logic and derive conclusions using forward chaining, backward chaining and truth-table entailment.

In [ ]:
# Real-world problem: Student scholarship eligibility
# Propositional symbols:
#   Attendance75   : student has at least 75% attendance
#   MarksAbove80   : student scored above 80% marks
#   Regular        : student is regular
#   Meritorious    : student is meritorious
#   LowIncome      : family income is low
#   NoBacklog      : student has no backlog
#   Eligible       : student is eligible for scholarship
#   Applied        : student has applied
#   Selected       : student is selected for scholarship

# Rules (Horn clauses):  (premises) -> conclusion
rules = [
    ({"Attendance75"}, "Regular"),
    ({"MarksAbove80", "NoBacklog"}, "Meritorious"),
    ({"Regular", "Meritorious"}, "Eligible"),
    ({"Regular", "LowIncome"}, "Eligible"),
    ({"Eligible", "Applied"}, "Selected"),
]

# Known facts about a particular student
facts = {"Attendance75", "MarksAbove80", "NoBacklog", "Applied"}

print("Knowledge Base Rules:")
for premises, conclusion in rules:
    print("  ", " AND ".join(sorted(premises)), "=>", conclusion)
print("\nInitial facts:", facts)

In [ ]:
# ---------- 1. Forward chaining (data-driven inference) ----------
def forward_chaining(rules, facts, query=None):
    known = set(facts)
    steps = []
    changed = True
    while changed:
        changed = False
        for premises, conclusion in rules:
            if premises <= known and conclusion not in known:
                known.add(conclusion)
                steps.append(f"{' AND '.join(sorted(premises))}  =>  {conclusion}")
                changed = True
                if query and conclusion == query:
                    return True, known, steps
    return (query in known if query else None), known, steps

result, derived, steps = forward_chaining(rules, facts, "Selected")
print("Forward chaining steps:")
for i, s in enumerate(steps, 1):
    print(f"  {i}. {s}")
print("\nIs 'Selected' derived? ->", result)
print("All derived facts:", derived)

In [ ]:
# ---------- 2. Backward chaining (goal-driven inference) ----------
def backward_chaining(goal, rules, facts, visited=None, depth=0):
    if visited is None:
        visited = set()
    indent = "  " * depth
    if goal in facts:
        print(f"{indent}{goal}: known fact")
        return True
    if goal in visited:          # avoid infinite loops
        return False
    visited = visited | {goal}
    for premises, conclusion in rules:
        if conclusion == goal:
            print(f"{indent}{goal}: try rule {sorted(premises)} => {goal}")
            if all(backward_chaining(p, rules, facts, visited, depth + 1) for p in premises):
                print(f"{indent}{goal}: PROVED")
                return True
    print(f"{indent}{goal}: cannot be proved")
    return False

print("Goal: Selected")
print("Result:", backward_chaining("Selected", rules, facts))

In [ ]:
# ---------- 3. Truth-table entailment check (KB |= query) ----------
from itertools import product

symbols = ["Attendance75", "MarksAbove80", "NoBacklog", "LowIncome",
           "Regular", "Meritorious", "Eligible", "Applied", "Selected"]

def kb_true(model):
    # every rule (premises -> conclusion) must hold, and all given facts must be true
    for premises, conclusion in rules:
        if all(model[p] for p in premises) and not model[conclusion]:
            return False
    return all(model[f] for f in facts)

def tt_entails(query):
    for values in product([True, False], repeat=len(symbols)):
        model = dict(zip(symbols, values))
        if kb_true(model) and not model[query]:
            return False          # found a model where KB true but query false
    return True

for q in ["Regular", "Meritorious", "Eligible", "Selected", "LowIncome"]:
    print(f"KB |= {q:12s} ->", tt_entails(q))

In [ ]:
# ---------- 4. First-Order Logic style inference (with variables) ----------
# FOL rules:
#   forall x: Attendance75(x) -> Regular(x)
#   forall x: MarksAbove80(x) AND NoBacklog(x) -> Meritorious(x)
#   forall x: Regular(x) AND Meritorious(x) -> Eligible(x)
#   forall x: Eligible(x) AND Applied(x) -> Selected(x)

fol_facts = {
    ("Attendance75", "Asha"), ("MarksAbove80", "Asha"), ("NoBacklog", "Asha"), ("Applied", "Asha"),
    ("Attendance75", "Ravi"), ("MarksAbove80", "Ravi"), ("Applied", "Ravi"),     # Ravi has a backlog
    ("Attendance75", "Meera"), ("MarksAbove80", "Meera"), ("NoBacklog", "Meera"), # Meera has not applied
}

fol_rules = [
    (["Attendance75"], "Regular"),
    (["MarksAbove80", "NoBacklog"], "Meritorious"),
    (["Regular", "Meritorious"], "Eligible"),
    (["Eligible", "Applied"], "Selected"),
]

people = sorted({p for _, p in fol_facts})
changed = True
while changed:
    changed = False
    for premises, conclusion in fol_rules:
        for x in people:                       # unify variable x with each constant
            if all((p, x) in fol_facts for p in premises) and (conclusion, x) not in fol_facts:
                fol_facts.add((conclusion, x))
                changed = True

for person in people:
    print(f"{person:6s} -> Eligible: {('Eligible', person) in fol_facts!s:5} | Selected: {('Selected', person) in fol_facts}")